# 03 — Deploy Services

In this notebook we deploy all services onto the Kubernetes cluster:

**Platform services** (`immich-platform` namespace):
- MinIO — S3-compatible object storage for MLflow artifacts
- MLflow — model registry and experiment tracker
- PostgreSQL — MLflow metadata database

**Immich** (`immich` namespace):
- PostgreSQL with pgvecto-rs — Immich database
- Redis — job queue and session cache
- Immich server — REST API and web UI
- Immich microservices — background processing (thumbnails, video transcode)
- Immich machine-learning — CLIP embeddings and face recognition

We use an **Ansible playbook** that:
1. Detects the external IP of node1 automatically
2. Generates all secrets at deploy time (never stored in Git)
3. Applies all K8s manifests
4. Patches services with the correct external IP


## Prerequisites

- `02_install_k8s.ipynb` must be complete with `failed=0`
- You need the floating IP from `01_provision.ipynb`


In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
export PYTHONUSERBASE=/work/.local

## Step 1: Confirm the cluster is healthy before deploying

In [ ]:
# runs in Chameleon Jupyter environment
# SSH into node1 and check cluster status
# Replace A.B.C.D with your floating IP
ssh -o StrictHostKeyChecking=no cc@A.B.C.D 'kubectl get nodes'

All 3 nodes should show `STATUS = Ready`. If any show `NotReady`, wait 2 minutes and retry.

## Step 2: Deploy platform services (MLflow + MinIO)

The Ansible playbook generates secrets, creates namespaces, and applies all manifests automatically.


In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
export PYTHONUSERBASE=/work/.local

cd /work/immich-iac/ansible
ansible-playbook -i inventory.yml deploy/deploy_platform.yml

## Step 3: Deploy Immich

In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
export PYTHONUSERBASE=/work/.local

cd /work/immich-iac/ansible
ansible-playbook -i inventory.yml deploy/deploy_immich.yml

## Step 4: Check service URLs

The playbooks above print the URLs at the end. They follow this pattern:
- **Immich:**  `http://A.B.C.D:2283`
- **MLflow:**  `http://A.B.C.D:8000`
- **MinIO console:** `http://A.B.C.D:9001`

Replace `A.B.C.D` with your floating IP and open each in a browser.

## Step 5: Wait for all pods to reach Running state


In [ ]:
# runs in Chameleon Jupyter environment
# Replace A.B.C.D with your floating IP
ssh -o StrictHostKeyChecking=no cc@A.B.C.D 'kubectl get pods -n immich'

In [ ]:
# runs in Chameleon Jupyter environment
# Replace A.B.C.D with your floating IP
ssh -o StrictHostKeyChecking=no cc@A.B.C.D 'kubectl get pods -n immich-platform'

All pods should show `Running`. It may take 3–5 minutes for all pods to start (images are being pulled).  
If any show `Pending` or `ContainerCreating`, wait and re-run the cells above.

---
**Continue to `04_verify.ipynb`**
